# JN 07: Dissolved Monks Transfer

This notebook calculates two measures of monastic dispersal intensity:
1. **Nearest Neighbor**: Small house income assigned to the nearest large house of the same order.
2. **Gravity Model**: Small house income assigned to the large house of the same order with the highest 'gravity' ($Income / Distance$).

It then joins these measures to the primary parish dataset (`northParishFlows.shp`) using IDW and 20km buffer metrics.

In [1]:
import pandas as pd
import numpy as np
import geopandas as gp
import shapely as sh
import os
import shutil
from pathlib import Path
from tqdm.auto import tqdm
tqdm.pandas()

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / 'Data' / 'Raw'
PROCESSED = PROJECT_ROOT / 'Data' / 'Processed'

BUFFER_M = 20_000
FLAT_RADIUS_M = 10_000

## 1. Load Monastic Data and Project to BNG

In [2]:
print('Loading National Archives Data...')
df = pd.read_csv(RAW / 'CSV/NationalArchivesData.csv')

df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df = df.dropna(subset=['latitude', 'longitude']).copy()

gdf = gp.GeoDataFrame(
    df, 
    geometry=gp.points_from_xy(df.longitude, df.latitude), 
    crs='epsg:4326'
).to_crs('epsg:27700')

sm_houses = gdf[gdf['smallHouse'] == 1].copy()
lg_houses = gdf[gdf['smallHouse'] == 0].copy()

print(f"Found {len(sm_houses)} small houses and {len(lg_houses)} large houses.")

Loading National Archives Data...
Found 608 small houses and 191 large houses.


## 2. Assignment Logic (Nearest vs Gravity)

In [3]:
print('Calculating nearest and gravity-based large houses...')

def assign_lg_house(row, lg_df):
    same_order_lg = lg_df[lg_df['order'] == row['order']].copy()
    if len(same_order_lg) == 0:
        return pd.Series([None, None], index=['nearest_name', 'gravity_name'])
    
    dist = same_order_lg.geometry.distance(row['geometry'])
    nearest_idx = dist.idxmin()
    
    # Gravity Model: Income_lg / Distance
    # Handle zero distance to avoid division by zero (unlikely but safe)
    dist_km = dist.replace(0, 0.001) / 1000.0
    gravity = same_order_lg['NAnetInc'] / dist_km
    gravity_idx = gravity.idxmax()
    
    return pd.Series([same_order_lg.loc[nearest_idx, 'name'], same_order_lg.loc[gravity_idx, 'name']], index=['nearest_name', 'gravity_name'])

assignments = sm_houses.progress_apply(assign_lg_house, axis=1, lg_df=lg_houses)
sm_houses = pd.concat([sm_houses, assignments], axis=1)

# Aggregate Nearest Neighbor
nn_transfer = sm_houses.dropna(subset=['nearest_name']).groupby('nearest_name')['NAnetInc'].sum().reset_index()
nn_transfer = nn_transfer.rename(columns={'nearest_name': 'name', 'NAnetInc': 'dissolved_L'})

# Aggregate Gravity Model
grav_transfer = sm_houses.dropna(subset=['gravity_name']).groupby('gravity_name')['NAnetInc'].sum().reset_index()
grav_transfer = grav_transfer.rename(columns={'gravity_name': 'name', 'NAnetInc': 'dissolved_L_g'})

# Merge back to large houses
lg_houses = lg_houses.merge(nn_transfer, on='name', how='left')
lg_houses = lg_houses.merge(grav_transfer, on='name', how='left')
lg_houses[['dissolved_L', 'dissolved_L_g']] = lg_houses[['dissolved_L', 'dissolved_L_g']].fillna(0)

print(f"Nearest Transfer: {lg_houses['dissolved_L'].sum():.2f}")
print(f"Gravity Transfer: {lg_houses['dissolved_L_g'].sum():.2f}")

Calculating nearest and gravity-based large houses...


  0%|          | 0/608 [00:00<?, ?it/s]

Nearest Transfer: 32902.00
Gravity Transfer: 32902.00


## 3. Parish Joining

In [4]:
print('Loading northParishFlows.shp...')
parish_path = PROCESSED / 'northParishFlows.shp'
parish_df = gp.read_file(parish_path)

parish_centroids = parish_df.geometry.centroid
cx = parish_centroids.x.values
cy = parish_centroids.y.values

for suffix, col in zip(['', '_g'], ['dissolved_L', 'dissolved_L_g']):
    active_lg = lg_houses[lg_houses[col] > 0].copy()
    if len(active_lg) == 0:
        parish_df[f'dis_L{suffix}_20'] = 0
        parish_df[f'dis_L{suffix}_w'] = 0.0
        continue
    
    # Buffer
    buf = active_lg.geometry.buffer(BUFFER_M).union_all()
    parish_df[f'dis_L{suffix}_20'] = parish_centroids.within(buf).astype(int)
    
    # IDW
    mx = active_lg.geometry.x.values
    my = active_lg.geometry.y.values
    vals = active_lg[col].values
    dist_m = np.sqrt((cx[:, np.newaxis] - mx[np.newaxis, :])**2 + (cy[:, np.newaxis] - my[np.newaxis, :])**2)
    weights = np.where(dist_m <= FLAT_RADIUS_M, 1.0, FLAT_RADIUS_M / dist_m)
    parish_df[f'dis_L{suffix}_w'] = (weights * vals[np.newaxis, :]).sum(axis=1)
    
    print(f"Metric {col} joined.")

Loading northParishFlows.shp...


Metric dissolved_L joined.
Metric dissolved_L_g joined.


## 4. Save and Overwrite with Backup

In [5]:
backup_path = PROCESSED / 'northParishFlows_pre_jn07.shp'
if not backup_path.exists():
    print('Creating backup...')
    for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg']:
        src = parish_path.with_suffix(ext)
        dst = backup_path.with_suffix(ext)
        if src.exists(): shutil.copy2(src, dst)

print('Saving to temp and swapping...')
temp_path = PROCESSED / 'northParishFlows_tmp.shp'
parish_df.to_file(temp_path)

for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg']:
    src = temp_path.with_suffix(ext)
    dst = parish_path.with_suffix(ext)
    if src.exists():
        try:
            if os.path.exists(dst): os.remove(dst)
            os.rename(src, dst)
        except PermissionError:
            print(f"Permission Error on {dst}")
print('Done!')

Saving to temp and swapping...


C:\Users\nicho\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Creating a 256th field, but some DBF readers might only support 255 fields
  ogr_write(


Permission Error on C:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\Data\Processed\northParishFlows.shp
Permission Error on C:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\Data\Processed\northParishFlows.dbf
Done!
